<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/01_chunk_documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 648, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (185/185), done.
remote: Total 648 (delta 118), reused 25 (delta 13), pack-reused 450 (from 2)
Receiving objects: 100% (648/648), 597.71 KiB | 3.25 MiB/s, done.
Resolving deltas: 100% (393/393), done.


In [25]:
%cd /content

/content


In [18]:
!pwd
!ls

/content/ML-Tech
app  data  metadata.xlsx  notebooks  README.md	src  tests


In [15]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [27]:
!pwd
!ls data/processed

/content/ML-Tech
chunks.json  passport_index.faiss


In [28]:
from pathlib import Path

%cd ML-Tech
raw_folder = Path("data/raw")

documents = []

for file_path in raw_folder.rglob("*.txt"):
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()

    documents.append({
        "filename": file_path.name,
        "content": text
    })

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(doc["filename"])

/content/ML-Tech/ML-Tech
Loaded 12 documents
استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي.txt
الاستحصال على رخصة سوق دولية.txt
استبدال رخصة سوق لبنانية لأجنبية.txt
الاستحصال على رخصة سوق بدل عن ضائع.txt
استبدال رخصة السوق العسكرية للعسكريين الموجودين في الخدمة الفعلية.txt
تجديد رخصة السوق.txt
Lost Passport or Stolen Passport.txt
Personal attendance required.txt
Certifying Passport.txt
Ex-porting Biometric Passport.txt
Biometric Passport.txt
Passport of adopted, born in special circumstances child or a citizen without a family name.txt


In [29]:
document_sections = {
    "Biometric Passport.txt": [
        "Requested documents:",
        "Remarks:",
        "Fees:",
        "NB:"
    ],

    "Lost Passport or Stolen Passport.txt": [
        "Lost Passport:",
        "Stolen Passport:",
        "NB:"
    ],

    "Ex-porting Biometric Passport.txt": [
        "Exporting a Lebanese passport",
        "Exporting a Foreign passport",
        "Lebanese or foreign passport shipped:",
        "For travel agencies that plan to ship passports:",
        "For the individual planning on shipping his passport with another traveler:",
        "Nb:"
    ],
    "Passport of adopted, born in special circumstances child or a citizen without a family name.txt":[
        "The requested documents",
        "Child born in special cirmustances",
        "A passport for a minor:",
        "A citizen without a family name"
    ],
    "Personal attendance required.txt":[
        "Personal attendance required",
        "Exemption from attendance",
        "Exemption from fees"
    ],
    "Certifying Passport.txt":[]
}

In [30]:
def extract_metadata(text):
    metadata = {
        "url": "",
        "title": "",
        "category": "",
        "keywords": ""
    }

    # Separate metadata from actual content
    if "Content:" in text:
        metadata_part, content = text.split("Content:", 1)
    else:
        metadata_part = ""
        content = text

    # Extract each metadata field
    for line in metadata_part.splitlines():
        line = line.strip()

        if line.startswith("URL:"):
            metadata["url"] = line.replace("URL:", "", 1).strip()

        elif line.startswith("Title:"):
            metadata["title"] = line.replace("Title:", "", 1).strip()

        elif line.startswith("Category:"):
            metadata["category"] = line.replace("Category:", "", 1).strip()

        elif line.startswith("Keywords:"):
            metadata["keywords"] = line.replace("Keywords:", "", 1).strip()

    return metadata, content.strip()

In [31]:
metadata, content = extract_metadata(documents[0]["content"])

print(metadata)
print(content)

{'url': 'https://tmo.gov.lb/web/panel/info/service-types/1', 'title': '"استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي(يستوجب حضور صاحب العلاقة شخصيا)(للاطلاع على المستندات المطلوبة دون الحاجة لحجز موعد مسبق)"', 'category': "Driver's Licence", 'keywords': "Driver's license, driving license, Lebanon, Lebanese, documents, appointment,taxi,cab, public, social security,رخصة سوق، لبنان، لبناني، المستندات ، عمومي، ضمان اجتماعي، موعد،"}
1
الوصف

لا يتطلب موعد
(يستوجب حضور صاحب العلاقة شخصيا)
1.1
استمارة تُطلب من قبل الصندوق الوطني للضمان الاجتماعي للمستفيدين من خدماته من فئة السائقين العموميين، وذلك للتأكّد من تجديدهم لرخصة السوق بصورة منتظمة.

2
المستندات المطلوبة:

2.1
طلب افادة من الضمان الاجتماعي

2.2
صورة عن رخصة السير العمومية

2.3
صورة عن الهوية


In [32]:
import re

def chunk_document(filename, text, headings,metadata):
    chunks = []

    # If there are no headings, keep the whole document
    if not headings:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": "Full Document",
            "text": text.strip()
        })
        return chunks

    # Build a regex from the headings
    pattern = "|".join(re.escape(h) for h in headings)

    # Split while keeping the headings
    parts = re.split(f"({pattern})", text)

    current_heading = None
    current_text = ""

    for part in parts:

        if part in headings:

            if current_heading is not None:
                chunks.append({
                    "document": filename,
                     "title":metadata["title"],
                    "url":metadata["url"],
                    "category":metadata["category"],
                    "keywords":metadata["keywords"],
                    "section": current_heading,
                    "text": current_text.strip()
                })

            current_heading = part.rstrip(":")
            current_text = ""

        else:
            current_text += part

    # Save the last chunk
    if current_heading is not None:
        chunks.append({
            "document": filename,
            "title":metadata["title"],
            "url":metadata["url"],
            "category":metadata["category"],
            "keywords":metadata["keywords"],
            "section": current_heading,
            "text": current_text.strip()

        })

    return chunks

In [33]:
all_chunks = []
for document in documents:
    filename = document["filename"]
    raw_text = document["content"]

    # remove URL, title, category, keywords, etc.
    metadata,clean_text = extract_metadata(raw_text)

    headings = document_sections.get(filename, [])

    chunks = chunk_document(
        filename,
        clean_text,
        headings,
        metadata
    )

    all_chunks.extend(chunks)

In [34]:
lost_chunk = None
stolen_chunk = None

for chunk in all_chunks:
    if chunk["section"] == "Lost Passport":
        lost_chunk = chunk

    elif chunk["section"] == "Stolen Passport":
        stolen_chunk = chunk

In [35]:
if lost_chunk and stolen_chunk:

    stolen_chunk["text"] = (
        "Lost passport procedure:\n"
        + lost_chunk["text"]
        + "\n\n"
        + "Additional information for a stolen passport:\n"
        + stolen_chunk["text"]
    )

In [36]:
import json
import os

os.makedirs("data/processed", exist_ok=True)

with open("data/processed/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=4, ensure_ascii=False)

In [37]:
import json

with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(json.dumps(chunks, indent=4, ensure_ascii=False))

[
    {
        "document": "استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي.txt",
        "title": "\"استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي(يستوجب حضور صاحب العلاقة شخصيا)(للاطلاع على المستندات المطلوبة دون الحاجة لحجز موعد مسبق)\"",
        "url": "https://tmo.gov.lb/web/panel/info/service-types/1",
        "category": "Driver's Licence",
        "keywords": "Driver's license, driving license, Lebanon, Lebanese, documents, appointment,taxi,cab, public, social security,رخصة سوق، لبنان، لبناني، المستندات ، عمومي، ضمان اجتماعي، موعد،",
        "section": "Full Document",
        "text": "1\nالوصف\n\nلا يتطلب موعد\n(يستوجب حضور صاحب العلاقة شخصيا)\n1.1\nاستمارة تُطلب من قبل الصندوق الوطني للضمان الاجتماعي للمستفيدين من خدماته من فئة السائقين العموميين، وذلك للتأكّد من تجديدهم لرخصة السوق بصورة منتظمة.\n\n2\nالمستندات المطلوبة:\n\n2.1\nطلب افادة من الضمان الاجتماعي\n\n2.2\nصورة عن رخصة السير العمومية\n\n2.3\nصورة عن الهوية"
    },
    {
        "document": "الا

In [38]:
!git config --global user.email "yarajnoun@gmail.com"
!git config --global user.name "yaranoun"

In [39]:
!git add data/processed/chunks.json
!git commit -m "Update document chunks"
!git pull --rebase origin main
!git push origin main

[main 37ef78e] Update document chunks
 1 file changed, 72 insertions(+), 188 deletions(-)
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Current branch main is up to date.
fatal: could not read Username for 'https://github.com': No such device or address


In [40]:
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)